# Medical Imaging Shortcut Detection - CheXpert Demo

This notebook demonstrates shortcut detection on **medical imaging data** using the CheXpert chest X-ray validation set (lightweight demo).

**What you'll learn:**
- How to load and process CheXpert metadata
- How to detect sex-based shortcuts in embeddings
- How to generate a comprehensive fairness report

**Note:** This demo uses synthetic embeddings to simulate ResNet50 features. For production use, extract real embeddings from your trained models.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from shortcut_detect import ShortcutDetector
from shortcut_detect import generate_linear_shortcut

print("✅ Imports successful!")

✅ Imports successful!


In [2]:
# Setup paths - works from any directory
notebook_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()

# Find project root
current = notebook_dir
for _ in range(5):
    if os.path.exists(os.path.join(current, 'data', 'valid.csv')):
        project_root = current
        break
    current = os.path.dirname(current)
else:
    project_root = os.path.abspath(os.path.join(notebook_dir, '..', '..'))

# Set paths
data_dir = os.path.join(project_root, 'data')
output_dir = os.path.join(project_root, 'examples', 'output_real_data')
csv_dir = os.path.join(output_dir, 'csv_exports')

# Create output directories
os.makedirs(output_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)

print(f"✅ Paths configured!")
print(f"   Data: {data_dir}")
print(f"   Output: {output_dir}")

✅ Paths configured!
   Data: /home/sebasmos/Desktop/Shortcut_Detect/data
   Output: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data


## 1. Load CheXpert Validation Data

Using validation set (234 samples) for fast demo.

In [3]:
# Load CheXpert validation data (small, fast)
data_path = os.path.join(data_dir, 'valid.csv')
df = pd.read_csv(data_path)

print(f"✅ Loaded {len(df)} samples")
print(f"\nDemographics:")
print(f"  - Sex: {df['Sex'].value_counts().to_dict()}")
print(f"  - Age: {df['Age'].min():.0f}-{df['Age'].max():.0f} years")

✅ Loaded 234 samples

Demographics:
  - Sex: {'Male': 128, 'Female': 106}
  - Age: 18-90 years


## 2. Prepare Sex Labels

In [4]:
# Create binary sex labels (0=Female, 1=Male)
sex_labels = df['Sex'].map({'Female': 0, 'Male': 1}).values

print(f"Sex labels:")
print(f"  - Female: {(sex_labels == 0).sum()} samples")
print(f"  - Male: {(sex_labels == 1).sum()} samples")

Sex labels:
  - Female: 106 samples
  - Male: 128 samples


## 3. Generate Lightweight Embeddings

Using 512-dim embeddings (lighter than full 2048-dim ResNet50) for faster computation.

In [5]:
# Generate lightweight embeddings (512-dim instead of 2048)
n_samples = len(df)
embedding_dim = 512  # Lighter than full ResNet50 (2048)
shortcut_dims = 5

embeddings, _ = generate_linear_shortcut(
    n_samples=n_samples,
    embedding_dim=embedding_dim,
    shortcut_dims=shortcut_dims,
    seed=42
)

labels = sex_labels

print(f"✅ Generated embeddings: {embeddings.shape}")
print(f"✅ Labels: {labels.shape}")

✅ Generated embeddings: (234, 512)
✅ Labels: (234,)


## 4. Run Shortcut Detection

Detect sex-based shortcuts using all three methods.

In [6]:
# Create detector with all methods
detector = ShortcutDetector(
    methods=['hbac', 'probe', 'statistical'],
    seed=42
)

# Fit on data
print("Running detection...")
detector.fit(embeddings, labels)
print("\n✅ Detection complete!")

Running detection...
Running HBAC detection...
Running probe-based detection...
Running statistical tests...
✅ Detection complete!

✅ Detection complete!


In [7]:
# View summary
print(detector.summary())

UNIFIED SHORTCUT DETECTION SUMMARY
Dataset: 234 samples, 512 dimensions
Methods used: hbac, probe, statistical

----------------------------------------------------------------------
HBAC (Clustering-based Detection)
----------------------------------------------------------------------
Shortcut detected: NO
Confidence: medium
Types: dimension_specific
Clusters found: 3

----------------------------------------------------------------------
Probe-based Detection
----------------------------------------------------------------------
Probe accuracy: 100.00%
⚠️  High probe accuracy suggests strong shortcut signal!

----------------------------------------------------------------------
Statistical Testing
----------------------------------------------------------------------
Comparisons performed: 1
Comparisons with significant features: 1
  [0_vs_1]: 27 significant features

OVERALL ASSESSMENT
🔴 HIGH RISK: Multiple methods detected shortcuts
  • Probe accuracy 100.0% (high risk)
  • 1 gro

## 5. Generate Comprehensive Report

ONE complete report with all results and visualizations.

In [8]:
# Generate comprehensive HTML report
report_path = os.path.join(output_dir, 'chexpert_sex_fairness_report.html')

detector.generate_report(
    output_path=report_path,
    format='html',
    include_visualizations=True,
    export_csv=True,
    csv_dir=csv_dir
)

# Close all matplotlib figures to free memory
plt.close('all')

print("\n" + "="*70)
print("✅ COMPREHENSIVE REPORT GENERATED")
print("="*70)
print(f"\n📊 HTML report: {report_path}")
print(f"📁 CSV exports: {csv_dir}/")
print("\nIncludes: HBAC, Probe, Statistical tests, and all visualizations")
print("\n✅ Done! All figures closed.")

Generating visualizations...
  Generating PCA plot...
  Generating t-SNE plot...
  Generating dimension importance plot...
  Generating cluster purity plot...
  Generating p-value heatmap...
✅ HTML report saved to: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/chexpert_sex_fairness_report.html
Exporting to CSV...
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/overall_summary.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/hbac_cluster_purities.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/hbac_dimension_importance.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/probe_predictions.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/probe_summary.csv
  ✅ Saved: /home/sebasmos/Desktop/Shortcut_Detect/examples/output_real_data/csv_exports/statistical_pvalues.cs

## 6. Optional: Generate PDF

In [ ]:
# Optional: Generate PDF report (requires weasyprint + system dependencies)
# On macOS: brew install pango gdk-pixbuf libffi
# On Ubuntu: apt-get install libpango-1.0-0 libpangocairo-1.0-0

pdf_path = os.path.join(output_dir, 'chexpert_sex_fairness_report.pdf')

try:
    detector.generate_report(
        output_path=pdf_path,
        format='pdf',
        include_visualizations=True
    )
    print(f"✅ PDF: {pdf_path}")
except OSError as e:
    if "libgobject" in str(e) or "library" in str(e).lower():
        print("⚠️  PDF generation skipped - missing system dependencies")
        print("   To enable PDF export, install required libraries:")
        print("   • macOS: brew install pango gdk-pixbuf libffi")
        print("   • Ubuntu: apt-get install libpango-1.0-0 libpangocairo-1.0-0")
    else:
        raise

# Close all matplotlib figures
plt.close('all')
print("✅ Done! All figures closed.")